In [9]:

import sys

import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy import text, bindparam

from nhs_waiting_lists.utils.proj_paths import find_project_root

project_root = find_project_root()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

DB_PATH = project_root / "data/nhs_consolidated.db"
DATA_DIR = "./data"

# conn = sqlite3.connect(DB_PATH)

conn = create_engine(f"sqlite:///{DB_PATH}")

In [10]:
PROVIDER_CODES = ['A4M8P']
TREATMENT_CODES = ('C_160')

In [11]:


query = text("""
             SELECT i.period,
                    i.provider_code                          as provider,
                    i.treatment_function_code                as treatment_code,
                    i.incomplete                             as incomplete,
                    i.incomplete_prev,
                    i.incomplete - i.incomplete_prev         AS inc_diff,
                    ((i.incomplete / i.incomplete_prev) - 1) AS frac_change,

                    i.admitted,
                    i.admitted_prev,
                    i.nonadmitted,
                    i.nonadmitted_prev,
                    i.new_periods,
                    i.new_periods_prev,
                    -- the accunting identity
                    (
                        i.incomplete
                            - (i.incomplete_prev
                                   + i.new_periods
                            - i.nonadmitted
                            - i.admitted
                            ))                               AS residual,
                    i.new_periods
                        - i.admitted
                        - i.nonadmitted
                                                             AS delta
             FROM metrics AS i
             WHERE i.provider_code IN :provider_codes
               AND treatment_function_code IN :treatment_codes
             ORDER BY provider ASC, treatment_function_code ASC, i.period ASC; \
             """).bindparams(
    bindparam('provider_codes', expanding=True),
    bindparam('treatment_codes', expanding=True)
)

df = pd.read_sql(query, conn, params={
    'provider_codes': PROVIDER_CODES,
    'treatment_codes': TREATMENT_CODES}
                 )  # type: ignore[arg-type]
df = df.assign(
    inc_diff=df["incomplete"] - df["incomplete_prev"],
)
df = df.assign(
    residual_check_diff=df['incomplete_prev'] - df['delta']
)
df = df.assign(
    residual_check=(df['residual'] - df['residual_check_diff']).abs().max()
)
df


,period,provider,treatment_code,incomplete,incomplete_prev,inc_diff,frac_change,admitted,admitted_prev,nonadmitted,nonadmitted_prev,new_periods,new_periods_prev,residual,delta,residual_check_diff,residual_check


In [12]:
df.query('provider == "R0B" and treatment_code == "C_320"')[
    [
        "period",
        "incomplete",
        "incomplete_prev",
        "inc_diff",
        "residual",
        "delta",
        "residual_check"
    ]
]

,period,incomplete,incomplete_prev,inc_diff,residual,delta,residual_check
